# Diagnóstico de Supuestos del Modelo
### Estadística Computacional para la Toma de Decisiones

**Magíster en Ciencia de Datos e Inteligencia Artificial**
Universidad Andrés Bello

*Maidana, J.P. (2026). Apunte — versión Jupyter Notebook.*

---

Versión **ejecutable** del apunte. La idea central de este cuaderno es el **contraste**: para cada supuesto verás (1) cómo se ve cuando **se cumple** —usando un modelo de referencia bien especificado— y (2) cómo se ve cuando **se viola**, con datos construidos para ese fin y el test formal detectándolo.

> 💡 Busca los comentarios **`👉 EXPERIMENTA`**: marcan parámetros que vale la pena cambiar para ver cómo reacciona el diagnóstico.

Sólo se usan `numpy`, `pandas`, `matplotlib`, `scipy` y `statsmodels` — todo viene preinstalado en Google Colab.

## Configuración del entorno

Importamos librerías, fijamos semilla, definimos estilo y una función auxiliar de uso frecuente (`residuos_vs_ajustados`). **Ejecuta esta celda primero.**

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

AZUL    = "#2c7fb8"   # bien / datos
VERDE   = "#41ab5d"   # correcto
NARANJA = "#fe9929"   # alerta
ROJO    = "#e34a33"   # violación / referencia cero

def residuos_vs_ajustados(ax, modelo, titulo=""):
    # Gráfico residuos vs valores ajustados (la herramienta de diagnóstico nº 1).
    ax.scatter(modelo.fittedvalues, modelo.resid, alpha=0.6, color=AZUL, edgecolor="none")
    ax.axhline(0, color=ROJO, ls="--", lw=1.5)
    ax.set_xlabel("Valores ajustados"); ax.set_ylabel("Residuos")
    if titulo:
        ax.set_title(titulo)

print("Entorno listo ✔")

---
# 1. Introducción

Ajustar un modelo de regresión lineal es fácil: unas pocas líneas de código te dan coeficientes, valores $p$ y un $R^2$ prometedor. Sin embargo, estos números son válidos **sólo si ciertos supuestos se cumplen**. Cuando los supuestos se violan, los coeficientes pueden ser sesgados, los errores estándar incorrectos y las conclusiones completamente erróneas.

El diagnóstico no es una formalidad académica. Es la parte que nos dice **si podemos confiar en nuestro modelo**. Un analista experimentado dedica tanto tiempo diagnosticando como ajustando. Este proceso requiere herramientas visuales y pruebas formales, junto con la habilidad para interpretar lo que nos dicen sobre nuestros datos.

> ### 📝 Lo que dominarás al finalizar
> Serás capaz de diagnosticar comprensivamente cualquier modelo de regresión lineal. Sabrás identificar violaciones de linealidad, homocedasticidad, normalidad e independencia. Podrás detectar *outliers* y observaciones influyentes que sesgan tus resultados. Desarrollarás el juicio para decidir cuándo las violaciones son menores y cuándo requieren acciones correctivas.

---
# 2. Los Cinco Supuestos y Su Importancia

## 2.1 Jerarquía de importancia

No todos los supuestos son igual de críticos:

- **Crítico — Linealidad.** Si la relación fundamental no es lineal, el modelo está mal especificado.
- **Serio — Homocedasticidad, Independencia, Multicolinealidad.** Los coeficientes pueden estar correctos, pero no podemos confiar en las inferencias.
- **Moderado — Normalidad.** En muestras grandes ($n > 30$), el impacto es menor gracias al Teorema Central del Límite.

## 2.2 Repaso de supuestos

| # | Supuesto | Formalmente | Si se viola |
|---|---|---|---|
| 1 | **Linealidad** | $E[y \mid \mathbf{X}] = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$ | Coeficientes sesgados (crítico) |
| 2 | **Independencia** | $\text{Cov}(\epsilon_i, \epsilon_j) = 0,\ i \neq j$ | Errores estándar incorrectos |
| 3 | **Homocedasticidad** | $\text{Var}(\epsilon_i \mid \mathbf{X}) = \sigma^2$ constante | Inferencias inválidas |
| 4 | **Normalidad** | $\epsilon_i \sim N(0, \sigma^2)$ | Afecta IC en muestras pequeñas |
| 5 | **No multicolinealidad** | $\mathbf{X}$ no correlacionadas entre sí | Coeficientes inestables |

## Preparación: un modelo de referencia

Para diagnosticar necesitamos un modelo. Empezamos con uno **bien especificado** —datos lineales, con errores normales, homocedásticos e independientes— que servirá como **referencia de lo que "bien" debe verse**. En las secciones siguientes lo compararemos contra modelos que violan cada supuesto.

Fíjate en el pie de la tabla `summary()`: `statsmodels` ya reporta ahí varios diagnósticos (Durbin-Watson, Omnibus, Jarque-Bera, Cond. No.).

In [ ]:
import statsmodels.api as sm

# Modelo de REFERENCIA: y lineal en x1,x2,x3 con ruido normal homocedástico.
rng = np.random.default_rng(0)
n = 200
datos = pd.DataFrame({
    "x1": rng.uniform(0, 10, n),
    "x2": rng.uniform(0, 10, n),
    "x3": rng.uniform(0, 10, n),
})
datos["y"] = 3 + 2*datos.x1 - 1.5*datos.x2 + 0.8*datos.x3 + rng.normal(0, 2, n)

X = sm.add_constant(datos[["x1", "x2", "x3"]])
modelo = sm.OLS(datos["y"], X).fit()

residuos = modelo.resid                 # usaremos estos nombres en todo el cuaderno
valores_ajustados = modelo.fittedvalues

print("Modelo de referencia ajustado.")
modelo.summary()

---
# 3. Diagnóstico de Linealidad

El supuesto de linealidad es el más fundamental. Si la relación no es lineal, estamos usando la herramienta equivocada.

## 3.1 Gráfico principal: residuos vs valores ajustados

Es la herramienta principal. Si el modelo captura correctamente la relación, los residuos deben ser **ruido aleatorio sin patrones**: una nube de puntos distribuida alrededor de cero.

Patrones problemáticos: formas de **U** o U invertida (relación cuadrática no capturada), **tendencias** lineales (variable importante omitida) o **bandas** horizontales (variable categórica faltante).

> ### 🧪 Interpretación de patrones
> - **Forma de U** → relación cuadrática no capturada. Solución: añadir término $x^2$.
> - **Tendencia lineal** → variable importante omitida. Solución: incluir variables adicionales.
> - **Bandas horizontales** → variable categórica faltante. Solución: incluir esa variable.
> - **Sin patrón** → ideal, el modelo captura bien la relación.

In [ ]:
# El modelo de referencia: residuos sin patrón (lo que queremos ver)
fig, ax = plt.subplots()
residuos_vs_ajustados(ax, modelo, "Modelo de referencia: nube sin estructura")
plt.tight_layout(); plt.show()

Ahora el **contraste**: ajustamos una recta a datos que en realidad son cuadráticos.

In [ ]:
# Modelo MAL especificado: relación cuadrática ajustada con una recta
rng_q = np.random.default_rng(11)
xq = rng_q.uniform(-3, 3, 200)
yq = 2 + 1.5 * xq**2 + rng_q.normal(0, 1.5, 200)     # 👉 EXPERIMENTA: cambia 1.5*xq**2
modelo_mal = sm.OLS(yq, sm.add_constant(pd.DataFrame({"x": xq}))).fit()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
residuos_vs_ajustados(axes[0], modelo,     "✓ Bien especificado (sin patrón)")
residuos_vs_ajustados(axes[1], modelo_mal, "✗ No lineal (forma de U clara)")
plt.tight_layout(); plt.show()

## 3.2 Test formal: RESET de Ramsey

El test RESET prueba si el modelo está correctamente especificado añadiendo potencias de los valores ajustados. Un valor $p$ pequeño sugiere que el modelo lineal **no** es adecuado.

In [ ]:
from statsmodels.stats.diagnostic import linear_reset

for nombre, mod in [("Referencia", modelo), ("Mal especificado", modelo_mal)]:
    p = linear_reset(mod, power=2, use_f=True).pvalue
    veredicto = "NO LINEALIDAD" if p < 0.05 else "linealidad OK"
    print(f"{nombre:18s} RESET  p = {p:.4f}   ->  {veredicto}")

## 3.3 Soluciones

Cuando detectamos no linealidad: **transformaciones** (log para relaciones exponenciales, raíz cuadrada, Box-Cox), **términos polinómicos** ($x^2$, $x^3$ —con cuidado de no sobreajustar), **interacciones** ($x_1 \times x_2$), o **modelos no lineales** (GAM, splines, machine learning).

Apliquemos la solución más directa al modelo cuadrático: añadir $x^2$.

In [ ]:
# Solución: añadir el término cuadrático
modelo_corregido = sm.OLS(yq, sm.add_constant(pd.DataFrame({"x": xq, "x2": xq**2}))).fit()

p_antes = linear_reset(modelo_mal, power=2, use_f=True).pvalue
p_desp  = linear_reset(modelo_corregido, power=2, use_f=True).pvalue
print(f"RESET  antes (recta): p = {p_antes:.4f}   ->  después (con x²): p = {p_desp:.4f}")
print(f"R²     antes: {modelo_mal.rsquared:.3f}   ->  después: {modelo_corregido.rsquared:.3f}")
print("Añadir x² resuelve la mala especificación.")

---
# 4. Diagnóstico de Normalidad de Residuos

El supuesto de normalidad se refiere a los **residuos**, no a las variables originales. Es el **menos crítico**, especialmente en muestras grandes.

## 4.1 Q-Q plots

El Q-Q plot grafica los cuantiles de los residuos estandarizados contra los de una normal estándar. Si los residuos son normales, los puntos caen sobre una **línea recta**. Desviaciones en las colas indican colas más pesadas/ligeras; una curva en **S** indica asimetría.

## 4.2 Tests formales

- **Shapiro-Wilk:** más poderoso en muestras pequeñas ($n < 2000$).
- **Jarque-Bera:** basado en asimetría y curtosis, apropiado para muestras grandes.

El problema: en muestras muy grandes detectan desviaciones triviales; en pequeñas pueden no detectar problemas serios. **Los gráficos suelen ser más informativos.**

In [ ]:
from scipy import stats

fig, ax = plt.subplots(figsize=(6, 5))
stats.probplot(residuos, dist="norm", plot=ax)
ax.get_lines()[0].set(marker="o", markerfacecolor=AZUL, markeredgecolor="none", alpha=0.6)
ax.get_lines()[1].set(color=ROJO, lw=2)
ax.set_title("Q-Q plot — modelo de referencia")
plt.tight_layout(); plt.show()

print(f"Shapiro-Wilk  p = {stats.shapiro(residuos)[1]:.4f}")
print(f"Jarque-Bera   p = {stats.jarque_bera(residuos)[1]:.4f}")

Contraste: residuos normales (referencia) vs residuos **sesgados**.

In [ ]:
# Residuos sesgados: errores con distribución exponencial (asimétrica)
rng_s = np.random.default_rng(5)
xs = rng_s.uniform(0, 10, 200)
es = rng_s.exponential(2, 200) - 2                  # asimétrico a la derecha, media ~0
modelo_sesgo = sm.OLS(1 + 2*xs + es, sm.add_constant(pd.DataFrame({"x": xs}))).fit()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, r, t in [(axes[0], residuos, "✓ Residuos normales (sobre la línea)"),
                 (axes[1], modelo_sesgo.resid, "✗ Residuos sesgados (curva en S)")]:
    stats.probplot(r, dist="norm", plot=ax)
    ax.get_lines()[0].set(marker="o", markerfacecolor=AZUL, markeredgecolor="none", alpha=0.6)
    ax.get_lines()[1].set(color=ROJO, lw=2)
    ax.set_title(t)
plt.tight_layout(); plt.show()

print(f"Shapiro normal:  p = {stats.shapiro(residuos)[1]:.4f}   (no rechaza normalidad)")
print(f"Shapiro sesgado: p = {stats.shapiro(modelo_sesgo.resid)[1]:.4f}   (rechaza: NO normal)")

> ### ❗ Normalidad: importante pero no fatal
> - **Muestras grandes ($n > 100$):** el TCL protege; desviaciones moderadas no invalidan conclusiones.
> - **Muestras pequeñas ($n < 30$):** la normalidad es más importante; violaciones serias son problemáticas.
> - **Para predicción:** no afecta predicciones puntuales, sólo los intervalos de predicción.
> - **Prioridad:** dedica más esfuerzo a linealidad y homocedasticidad que a normalidad.

**Soluciones:** transformaciones logarítmicas para corregir asimetría; identificar y tratar *outliers*; usar métodos robustos o *bootstrap* para intervalos sin asumir normalidad.

---
# 5. Diagnóstico de Homocedasticidad

La homocedasticidad requiere **varianza constante** de los errores. Su violación (heterocedasticidad) es muy común en datos reales.

## 5.1 Detección visual

En el gráfico de residuos vs ajustados, la dispersión debe ser constante. La heterocedasticidad se manifiesta como un **patrón de abanico**. El *Scale-Location plot* —que grafica $\sqrt{|\text{residuos estandarizados}|}$ vs valores ajustados— muestra una **banda horizontal** cuando hay homocedasticidad.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

# Scale-Location del modelo de referencia
resid_std = residuos / residuos.std()
fig, ax = plt.subplots()
ax.scatter(valores_ajustados, np.sqrt(np.abs(resid_std)), alpha=0.6, color=AZUL, edgecolor="none")
ax.set_xlabel("Valores ajustados"); ax.set_ylabel(r"$\sqrt{|\,residuos\ estandarizados\,|}$")
ax.set_title("Scale-Location — referencia (banda horizontal)")
plt.tight_layout(); plt.show()

print(f"Breusch-Pagan  p = {het_breuschpagan(residuos, modelo.model.exog)[1]:.4f}")

## 5.2 Tests formales

- **Breusch-Pagan:** prueba si la varianza depende linealmente de las variables independientes.
- **White:** más general, detecta formas complejas de heterocedasticidad.

Ambos prueban $H_0$: homocedasticidad. Un valor $p$ pequeño indica heterocedasticidad. Veamos el contraste:

In [ ]:
# Datos heterocedásticos: la dispersión del error crece con x
rng_h = np.random.default_rng(3)
xh = rng_h.uniform(1, 10, 200)
yh = 2 + 1.5*xh + rng_h.normal(0, 0.6*xh)            # 👉 EXPERIMENTA: 0.6*xh -> 0.1 (homocedástico)
modelo_hetero = sm.OLS(yh, sm.add_constant(pd.DataFrame({"x": xh}))).fit()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
residuos_vs_ajustados(axes[0], modelo,        "✓ Homocedástico (dispersión constante)")
residuos_vs_ajustados(axes[1], modelo_hetero, "✗ Heterocedástico (abanico)")
plt.tight_layout(); plt.show()

print(f"BP referencia:     p = {het_breuschpagan(residuos, modelo.model.exog)[1]:.4f}")
print(f"BP heterocedástico:p = {het_breuschpagan(modelo_hetero.resid, modelo_hetero.model.exog)[1]:.4f}")

## 5.3 Consecuencias y soluciones

La heterocedasticidad **no sesga los coeficientes** pero invalida los errores estándar, haciendo los valores $p$ poco confiables.

**Solución más práctica:** errores estándar **robustos (HC3)**, que corrigen los errores estándar sin cambiar los coeficientes.

In [ ]:
# Errores estándar robustos HC3 sobre el modelo heterocedástico
modelo_hc3 = sm.OLS(yh, sm.add_constant(pd.DataFrame({"x": xh}))).fit(cov_type="HC3")

comp = pd.DataFrame({
    "Coeficiente": modelo_hetero.params,
    "SE clásico":  modelo_hetero.bse,
    "p clásico":   modelo_hetero.pvalues,
    "SE HC3":      modelo_hc3.bse,
    "p HC3":       modelo_hc3.pvalues,
})
print(comp.round(4).to_string())
print("\nLos coeficientes son idénticos; los errores estándar (y por tanto los p) cambian.")

**Alternativas:** transformar $Y$ (logaritmo), mínimos cuadrados ponderados (WLS) o transformación Box-Cox.

---
# 6. Diagnóstico de Independencia

El supuesto de independencia requiere que los errores **no estén correlacionados**. Es crítico en series temporales, datos espaciales o datos con estructura jerárquica.

## 6.1 ¿Cuándo preocuparse?

- **Series temporales:** observaciones ordenadas en el tiempo tienen errores correlacionados.
- **Datos espaciales:** observaciones geográficamente cercanas se parecen.
- **Datos agrupados:** mediciones repetidas del mismo individuo o grupo.

Si tus datos son una muestra aleatoria simple, la independencia usualmente no es problemática.

## 6.2 Test de Durbin-Watson

El estadístico DW va de 0 a 4; valores cercanos a **2** indican independencia. DW $< 1.5$ sugiere autocorrelación **positiva**; DW $> 2.5$, autocorrelación **negativa**.

In [ ]:
from statsmodels.stats.stattools import durbin_watson

# Errores autocorrelacionados (AR1) en datos ordenados en el tiempo
rng_a = np.random.default_rng(1)
n_t = 200
e = np.zeros(n_t); e[0] = rng_a.normal()
for t in range(1, n_t):
    e[t] = 0.8 * e[t-1] + rng_a.normal()             # 👉 EXPERIMENTA: 0.8 -> 0.0 (independiente)
tiempo = np.arange(n_t)
modelo_ar = sm.OLS(5 + 0.3*tiempo + e, sm.add_constant(pd.DataFrame({"t": tiempo}))).fit()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(np.arange(len(residuos)), residuos, ".", color=AZUL, alpha=0.7)
axes[0].axhline(0, color=ROJO, ls="--")
axes[0].set_title("✓ Independientes (sin patrón temporal)")
axes[0].set_xlabel("Orden"); axes[0].set_ylabel("Residuo")
axes[1].plot(tiempo, modelo_ar.resid, ".-", color=NARANJA, alpha=0.7, lw=0.8)
axes[1].axhline(0, color=ROJO, ls="--")
axes[1].set_title("✗ Autocorrelacionados (rachas largas)")
axes[1].set_xlabel("Orden (tiempo)"); axes[1].set_ylabel("Residuo")
plt.tight_layout(); plt.show()

print(f"Durbin-Watson referencia:        {durbin_watson(residuos):.3f}  (≈ 2: independiente)")
print(f"Durbin-Watson autocorrelacionado:{durbin_watson(modelo_ar.resid):.3f}  (< 1.5: autocorrelación +)")

## 6.3 Soluciones

Incluir variables omitidas (tendencias, estacionalidad), usar modelos de series temporales (ARIMA), aplicar errores estándar **HAC** (Newey-West), o para datos espaciales/jerárquicos usar modelos que incorporan la estructura de dependencia.

---
# 7. Identificación de Observaciones Influyentes

Un solo *outlier* extremo puede cambiar dramáticamente los coeficientes, llevándonos a conclusiones erróneas.

## 7.1 Conceptos clave

- **Outlier:** observación con residuo grande; el modelo predice mal su valor.
- **Observación influyente:** si se elimina, cambia sustancialmente los coeficientes.
- **Leverage (apalancamiento):** qué tan extrema es una observación en el espacio de $X$. Alto leverage no es necesariamente problemático, pero combinado con alto residuo sí lo es.

## 7.2 Métricas principales

- **Residuos studentizados:** estandarizados considerando el leverage. Valores absolutos $> 2$ o $3$ son candidatos a *outliers*.
- **Distancia de Cook:** combina residuo y leverage; mide cuánto cambiarían las predicciones al eliminar la observación. $D_i > 0.5$ es notable; $D_i > 1$, muy influyente.

$$ D_i = \frac{r_i^2}{p+1} \cdot \frac{h_{ii}}{1 - h_{ii}} $$

In [ ]:
from statsmodels.stats.outliers_influence import OLSInfluence

# Datos limpios + UN punto influyente inyectado (leverage alto y fuera de la recta)
rng_i = np.random.default_rng(1)
nb = 60
xb = rng_i.uniform(0, 10, nb); yb = 2 + 1.5*xb + rng_i.normal(0, 1, nb)
xb = np.append(xb, 20.0); yb = np.append(yb, 5.0)    # 👉 EXPERIMENTA: mueve este punto (20, 5)
modelo_infl = sm.OLS(yb, sm.add_constant(pd.DataFrame({"x": xb}))).fit()

influence = OLSInfluence(modelo_infl)
leverage = np.asarray(influence.hat_matrix_diag)
residuos_student = np.asarray(influence.resid_studentized_external)
cooks_d = np.asarray(influence.cooks_distance[0])

influyentes = np.where(cooks_d > 0.5)[0]
print(f"Observaciones influyentes (Cook's D > 0.5): {influyentes.tolist()}")
print(f"Cook's D máximo: {cooks_d.max():.2f}  (observación {cooks_d.argmax()}, "
      f"la última = el punto inyectado)")

In [ ]:
# Visualización del impacto del punto influyente
modelo_sin = sm.OLS(yb[:-1], sm.add_constant(pd.DataFrame({"x": xb[:-1]}))).fit()
xg = np.linspace(0, 21, 50)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (1) datos + rectas con y sin el punto
axes[0].scatter(xb[:-1], yb[:-1], color=AZUL, alpha=0.6, label="Datos")
axes[0].scatter(xb[-1], yb[-1], color=ROJO, s=130, zorder=5, label="Punto influyente")
axes[0].plot(xg, modelo_infl.params.iloc[0] + modelo_infl.params.iloc[1]*xg,
             color=ROJO, lw=2, label="Recta CON el punto")
axes[0].plot(xg, modelo_sin.params.iloc[0] + modelo_sin.params.iloc[1]*xg,
             color=VERDE, lw=2, ls="--", label="Recta SIN el punto")
axes[0].set_title("Un punto inclina toda la recta"); axes[0].legend(fontsize=8)
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")

# (2) Distancia de Cook
axes[1].bar(range(len(cooks_d)), cooks_d, color=AZUL)
axes[1].axhline(0.5, color=ROJO, ls="--", label="umbral 0.5")
axes[1].set_title("Distancia de Cook"); axes[1].set_xlabel("Observación"); axes[1].legend(fontsize=8)

# (3) Influence plot: leverage vs residuo studentizado
axes[2].scatter(leverage, residuos_student, alpha=0.6, color=AZUL)
axes[2].scatter(leverage[-1], residuos_student[-1], color=ROJO, s=130, zorder=5)
axes[2].axhline(0, color="gray", lw=0.8)
axes[2].axhline(3, color=ROJO, ls=":"); axes[2].axhline(-3, color=ROJO, ls=":")
axes[2].set_xlabel("Leverage"); axes[2].set_ylabel("Residuo studentizado")
axes[2].set_title("Influence plot")
plt.tight_layout(); plt.show()

## 7.3 ¿Qué hacer? — Análisis de sensibilidad

In [ ]:
# Comparar coeficientes con y sin el punto influyente
print("Análisis de sensibilidad — pendiente estimada:")
print(f"  CON el punto influyente: {modelo_infl.params.iloc[1]:.3f}")
print(f"  SIN el punto influyente: {modelo_sin.params.iloc[1]:.3f}")
print(f"  (la pendiente verdadera es 1.5; el punto la distorsiona casi a la mitad)")

**Qué hacer ante influyentes:** *investigar* (¿errores de medición? corrige o elimina; ¿valores legítimos pero extremos? pueden ser informativos), *análisis de sensibilidad* (ajusta con y sin el punto; si las conclusiones coinciden, procede con confianza) y *reportar honestamente* (si eliminas observaciones, dilo explícitamente).

> ### 🚨 No elimines outliers mecánicamente
> Los *outliers* pueden ser informativos y pueden indicar problemas del modelo. Eliminar *outliers* basándote en el mismo modelo que luego interpretarás **introduce sesgo**.
>
> Práctica recomendada: identifica, investiga caso por caso, corrige errores obvios, y reporta resultados **con y sin** los *outliers* legítimos.

---
# 8. Flujo de Diagnóstico Completo

Un diagnóstico completo debe seguir un orden sistemático para maximizar eficiencia.

## 8.1 Checklist

| Paso | Supuesto | Herramientas principales |
|---|---|---|
| 1 | Linealidad | Residuos vs ajustados, Test RESET |
| 2 | Homocedasticidad | Residuos vs ajustados, BP test |
| 3 | Independencia | Serie temporal, Durbin-Watson |
| 4 | Normalidad | Q-Q plot, Shapiro-Wilk |
| 5 | Multicolinealidad | VIF |
| 6 | Influyentes | Cook's D, Leverage |

## 8.2 Código de diagnóstico rápido

Empaquetamos los cuatro gráficos clave en una función reutilizable y la aplicamos al modelo de referencia (que, por construcción, se ve sano).

In [ ]:
def diagnostico_completo(modelo):
    '''Genera los 4 gráficos de diagnóstico estándar de un modelo OLS de statsmodels.'''
    res = modelo.resid
    fit = modelo.fittedvalues
    cooks = np.asarray(OLSInfluence(modelo).cooks_distance[0])

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    # Residuos vs Ajustados
    axes[0, 0].scatter(fit, res, alpha=0.5, color=AZUL, edgecolor="none")
    axes[0, 0].axhline(0, color=ROJO, ls="--")
    axes[0, 0].set_title("Residuos vs Ajustados")
    axes[0, 0].set_xlabel("Ajustados"); axes[0, 0].set_ylabel("Residuos")

    # Q-Q Plot
    stats.probplot(res, dist="norm", plot=axes[0, 1])
    axes[0, 1].get_lines()[0].set(marker="o", markerfacecolor=AZUL, markeredgecolor="none", alpha=0.5)
    axes[0, 1].get_lines()[1].set(color=ROJO, lw=2)
    axes[0, 1].set_title("Q-Q Plot")

    # Scale-Location
    axes[1, 0].scatter(fit, np.sqrt(np.abs(res / res.std())), alpha=0.5, color=AZUL, edgecolor="none")
    axes[1, 0].set_title("Scale-Location")
    axes[1, 0].set_xlabel("Ajustados"); axes[1, 0].set_ylabel(r"$\sqrt{|res.\ est.|}$")

    # Distancia de Cook
    axes[1, 1].bar(range(len(cooks)), cooks, color=AZUL)
    axes[1, 1].axhline(0.5, color=ROJO, ls="--")
    axes[1, 1].set_title("Distancia de Cook")
    axes[1, 1].set_xlabel("Observación")

    plt.tight_layout(); plt.show()

diagnostico_completo(modelo)

Y completamos el paso 5 del checklist (multicolinealidad) calculando el **VIF** de los predictores:

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = modelo.model.exog
nombres = modelo.model.exog_names
vif = pd.DataFrame({
    "Variable": nombres,
    "VIF": [variance_inflation_factor(X_vif, i) for i in range(X_vif.shape[1])],
})
print(vif[vif.Variable != "const"].round(2).to_string(index=False))
print("\nVIF cercanos a 1  ->  sin multicolinealidad (las X son independientes).")
print("Regla práctica: VIF > 5 es preocupante; VIF > 10 indica multicolinealidad severa.")

## 8.3 Decisiones post-diagnóstico

| Problema | Acción inmediata | Alternativas |
|---|---|---|
| No linealidad | Transformar, añadir términos | Modelos no lineales |
| Heterocedasticidad | Errores robustos (HC3) | Transformar $Y$, WLS |
| Autocorrelación | Incluir variables omitidas | HAC, modelos ARIMA |
| Falta normalidad | Si $n>100$, continuar | Bootstrap, regresión robusta |
| Multicolinealidad | Eliminar variable redundante | Ridge, PCA |
| Influyentes | Investigar, sensibilidad | Regresión robusta |

## 8.4 Mejores prácticas

Diagnostica **siempre** cada modelo. **Prioriza** problemas (linealidad primero). Usa gráficos y tests como **complementarios**. **Documenta** tu proceso. Haz **análisis de sensibilidad** si hay cambios importantes. Sé **honesto** sobre las limitaciones en tu reporte.

> ### 💡 Cierre
> El diagnóstico de supuestos no es una formalidad que podamos omitir. Es el proceso que nos dice si podemos confiar en nuestro modelo. Un analista experto dedica tanto tiempo diagnosticando como ajustando. Con las herramientas presentadas en este apunte, estás equipado para diagnosticar modelos rigurosamente, identificar problemas y tomar decisiones informadas. Esta habilidad te distinguirá como un científico de datos cuidadoso y confiable.

---
## Referencias bibliográficas

- Belsley, D. A., Kuh, E., & Welsch, R. E. (1980). *Regression diagnostics: Identifying influential data and sources of collinearity*. Wiley.
- Cook, R. D., & Weisberg, S. (1982). *Residuals and influence in regression*. Chapman and Hall.
- Greene, W. H. (2018). *Econometric analysis* (8th ed.). Pearson.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical learning* (2nd ed.). Springer.
- White, H. (1980). A heteroskedasticity-consistent covariance matrix estimator and a direct test for heteroskedasticity. *Econometrica*, 48(4), 817–838.